# Lekcija 7: Mjere udaljenosti

„Udaljenost” na mreži piksela nije tako jednostavna kako se čini. Razmotrit ćemo tri različita pristupa:

1. **Metrike udaljenosti između dviju točaka** (Manhattanska, šahovska i euklidska) &mdash; mjere koliko su dva piksela međusobno udaljena.
2. **Procjene duljine krivulje** (Freemanova i poboljšana ponderirana procjena) &mdash; mjere duljinu *digitalizirane krivulje*, primjerice ruba područja, na temelju njezina lančanog koda.
3. **Chamferova transformacija udaljenosti** &mdash; brza aproksimacija na razini cijele slike koja svakom pikselu pridružuje udaljenost do najbližeg člana zadanog skupa, primjerice objekta, izračunata u dva rasterska prolaza.


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Prvi dio: Metrike udaljenosti između dviju točaka

Za dva piksela $p=(x_1,y_1)$ i $q=(x_2,y_2)$ definirajmo $d_x = x_2-x_1$ i $d_y=y_2-y_1$. Tri uobičajene metrike su:

- **Manhattanska udaljenost / udaljenost gradskih blokova ($L_1$)**: udaljenost kada je kretanje dopušteno samo duž osi mreže, poput hodanja ulicama između gradskih blokova.

    $D_4(p,q) = |d_x| + |d_y|$
- **Šahovska udaljenost ($L_\infty$)**: broj poteza kralja na šahovskoj ploči; dijagonalni koraci nemaju dodatni trošak.

    $D_8(p,q) = \max(|d_x|,\,|d_y|)$
- **Euklidska udaljenost ($L_2$)**: uobičajena udaljenost duž ravne crte.

    $D_E(p,q) = \sqrt{d_x^2+d_y^2}$

Nazivi $D_4$ i $D_8$ proizlaze iz toga što svaka predstavlja duljinu najkraćeg puta kada su dopušteni samo pomaci prema 4-povezanosti (duž osi) ili 8-povezanosti (duž osi i dijagonalno), uz jedinični trošak svakog pomaka.


In [ ]:
def manhattan(dx, dy):
    return np.abs(dx) + np.abs(dy)

def chessboard(dx, dy):
    return np.maximum(np.abs(dx), np.abs(dy))

def euclidean(dx, dy):
    return np.sqrt(dx**2 + dy**2)

### Prikaz triju metrika pomoću polja udaljenosti

Za svaki piksel u mreži izračunavamo njegovu udaljenost do središnjeg piksela prema svakoj metrici i prikazujemo je kao sliku. Metrike se podudaraju samo duž osi — na svim ostalim mjestima daju različite vrijednosti.


In [ ]:
size = 101
half = size // 2
yy, xx = np.mgrid[-half:half + 1, -half:half + 1]

fields = {
    'Manhattan ($D_4$)': manhattan(xx, yy),
    'Chessboard ($D_8$)': chessboard(xx, yy),
    'Euclidean': euclidean(xx, yy),
}

# Compute a single vmin/vmax to prevent each imshow() from auto-scaling 
# (Manhattan's max is 100, Euclidean's is ~71, chessboard's is 50).
vmax = max(field.max() for field in fields.values())

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (name, field) in zip(axes, fields.items()):
    im = ax.imshow(field, cmap='viridis', vmin=0, vmax=vmax)
    ax.contour(field, levels=8, colors='white', linewidths=0.5)
    ax.set_title(name)
    ax.axis('off')
plt.tight_layout()
plt.show()

Linije jednake udaljenosti otkrivaju karakterističan oblik svake metrike: Manhattanska udaljenost stvara rombove, šahovska kvadrate, a euklidska kružnice — oblik koji većina ljudi intuitivno povezuje s jednakom udaljenošću od središta.


## Drugi dio: Procjena duljine krivulje iz lančanog koda

Pretpostavimo sada da želimo procijeniti duljinu *krivulje*, a ne samo udaljenost između dviju točaka. Izradimo testnu sliku kružnice poznatog polumjera ($60$) i poznatog opsega ($120 \pi = 377$). Primijetite da je broj rubnih piksela ($336$) loša procjena opsega jer je $336 \neq 377$.


In [ ]:
radius = 60
img = np.zeros((150, 150), dtype=np.uint8)
cv2.circle(img, (75, 75), radius, 255, 1)

contours, _ = cv2.findContours(img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
pts = contours[0][:, 0, :]

plt.imshow(img, cmap='gray')
plt.title(f'Digitized circle, radius={radius}, {len(pts)} boundary pixels')
plt.axis('off')
plt.show()

**Freemanov lančani kod** opisuje digitalnu krivulju kao niz jediničnih koraka u 8 mogućih smjerova (0–7, međusobno razmaknutih za 45°). Četiri od osam smjerova imaju *parne* oznake (vodoravni i okomiti, stvarne duljine 1), a četiri *neparne* oznake (dijagonalni, stvarne duljine $\sqrt{2}$).

Ako lančani kod krivulje sadrži $N_e$ parnih koraka i $N_o$ neparnih koraka, **Freemanova procjena** njezine duljine jednostavno je

$$L_{\text{Freeman}} = N_e + \sqrt{2}\,N_o$$

To zvuči ispravno, ali sustavno *precjenjuje* duljinu glatkih krivulja: rub stvarne kružnice nakon digitalizacije sadrži mnogo kratkih cik-cak koraka nalik stepenicama, čiji je zbroj veći od stvarne duljine luka.

Za rješavanje tog problema točne težine $(1, \sqrt{2})$ mogu se zamijeniti težinama $(a, b)$ prilagođenima tako da minimiziraju prosječnu pogrešku na velikom broju digitaliziranih krivulja — ideja koja se ponekad povezuje s Kimurinom korekcijom duljine lančanog koda. Često navođene vrijednosti, koje se malo razlikuju ovisno o izvoru, približno su $a \approx 0.948$ i $b \approx 1.343$:

$$L_{\text{corrected}} = a\,N_e + b\,N_o$$


In [ ]:
steps = np.diff(np.vstack([pts, pts[:1]]), axis=0)  # step from each boundary pixel to the next
is_diagonal = (steps[:, 0] != 0) & (steps[:, 1] != 0)

N_o = int(np.sum(is_diagonal))    # odd (diagonal) steps
N_e = int(np.sum(~is_diagonal))   # even (axis-aligned) steps

true_length = 2 * np.pi * radius
L_freeman = N_e + np.sqrt(2) * N_o

a, b = 0.948, 1.343
L_corrected = a * N_e + b * N_o

print(f'N_e (even/axis steps)     = {N_e}')
print(f'N_o (odd/diagonal steps)  = {N_o}')
print(f'number of boundary pixels = {len(pts):.1f}')
print(f'true circumference        = {true_length:.1f}')
print(f'Freeman estimate          = {L_freeman:.1f}  (error {100*(L_freeman-true_length)/true_length:+.1f}%)')
print(f'Corrected estimate        = {L_corrected:.1f}  (error {100*(L_corrected-true_length)/true_length:+.1f}%)')

Korigirane težine daju procjenu znatno bližu stvarnom opsegu, pri čemu je dovoljno samo prebrojiti parne i neparne korake — korisno kada želite računalno jednostavnu procjenu duljine izravno iz lančanog koda.


## Treći dio: Chamferova transformacija udaljenosti

Za razliku od metrika iz prvog dijela, koje mjere udaljenost između dvaju određenih piksela, **transformacija udaljenosti** *svakom* pikselu binarne slike pridružuje udaljenost do najbližeg člana zadanog skupa piksela — obično objekta ili pozadine. Korisna je za određivanje kostura oblika, usporedbu oblika i planiranje putanje. *Točan* izračun euklidske udaljenosti zahtijeva usporedbu svakog piksela sa svakim pikselom tog skupa, što je sporo, ili pažljivije osmišljen algoritam.

**Chamferova transformacija udaljenosti** brza je aproksimacija: umjesto jedne globalne pretrage prenosi lokalne udaljenosti kroz sliku u samo dva rasterska prolaza (od gornjeg lijevog do donjeg desnog kuta, pa u suprotnom smjeru), dodajući mali fiksni trošak za svaki korak do susjeda. Cjelobrojne težine $a=3$ za susjede duž osi i $b=4$ za dijagonalne susjede, uz dijeljenje rezultata s 3 na kraju, daju klasičnu **Chamferovu udaljenost 3–4**, dobru i računalno jednostavnu aproksimaciju euklidske udaljenosti. Ideja je ista kao kod Freemanovih i korigiranih težina lančanog koda iz drugog dijela, ali se primjenjuje na svaki piksel umjesto samo na rub.

Funkcija `cv2.distanceTransform` iz biblioteke OpenCV s parametrom `DIST_L2` izračunava Chamferovu udaljenost. Parametar `maskSize` određuje kvalitetu aproksimacije: veća maska daje točniji rezultat, ali zahtijeva više računanja.


In [ ]:
img = np.zeros((100, 100), dtype=np.uint8)
cv2.circle(img, (50, 50), 10, 255, -1)
img = 255 - img  # we will compute distance from each background pixel to the nearest foreground (nonzero) pixel

im_dist3 = cv2.distanceTransform(img, cv2.DIST_L2, maskSize = 3) # (3,4) chamfer algorithm 
im_dist5 = cv2.distanceTransform(img, cv2.DIST_L2, maskSize = 5) # more accurate algorithm

error = im_dist3 - im_dist5
print(f'max error  = {np.abs(error).max():.3f} pixels')
print(f'mean error = {np.abs(error).mean():.3f} pixels')

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, field, title in zip(
    axes,
    [im_dist3, im_dist5, error],
    ['Chamfer (3-4) approx.', 'More accurate estimate', 'Difference'],
):
    im = ax.imshow(field, cmap='viridis' if title != 'Difference' else 'coolwarm')
    ax.set_title(title, fontsize=10)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

Chamferova transformacija aproksimira točnu transformaciju euklidske udaljenosti, koristeći dva brza linearna prolaza umjesto računalno zahtjevne pretrage najbližeg susjeda.


### Zadaci

1. Ponovite pokus s kružnicom za nekoliko različitih polumjera, primjerice 10, 20, 40 i 80. Ostaje li postotna pogreška Freemanove procjene približno stalna ili se smanjuje kako kružnica raste? Zašto bi to moglo biti tako?
2. Šahovska udaljenost $D_8$ točna je duljina najkraćeg puta kada su dopušteni dijagonalni pomaci uz trošak 1. Možete li izraditi izmijenjenu metriku nalik šahovskoj, s troškom dijagonalnog koraka $\sqrt{2}$ umjesto 1, koja se ponaša kao gruba lokalna inačica prethodnih procjena duljine iz lančanog koda? Usporedite je sa stvarnom euklidskom udaljenošću za nekoliko točaka.
